# Lab Day 1 — Xây dựng mạng nơ-ron và thí nghiệm huấn luyện (Forest CoverType)

Notebook này chạy được từ đầu đến cuối (*Restart & Run All*) khi đặt tại `submission_<MSSV>/code/`, cùng cấp với `data/` và `scripts/` của repo (có thể ở thư mục cha, ô đầu tự tìm).
Mọi kết quả ghi vào `../figures/`, `../results/`, `../experiments.xlsx`, `../predictions_eval.csv`, `../eval_result.json`.

**Nguyên tắc:** chọn mọi cấu hình CHỈ bằng validation (20% của train, phân tầng, seed 42). Tập `eval` chỉ dùng ở Part 4, cho baseline và cấu hình cuối cùng.
Mỗi thí nghiệm: *dự đoán trước* → đổi **một** yếu tố → chạy → *đối chiếu*. Mọi lần chạy đều qua một hàm duy nhất `run_experiment(cfg, data)` (`train.py`).

**Chạy trên Colab/Kaggle:** tải nguyên repo (có `data/`, `scripts/`, `submission_<MSSV>/`) lên Drive hoặc giải nén vào `/content`, rồi `%cd` vào `submission_<MSSV>/code` trước khi chạy ô đầu. Bật GPU (T4). Nếu Colab ngắt giữa chừng, chạy lại: các lần chạy đã có `results/<exp_id>.json` sẽ được nạp lại (cờ `RESUME`), riêng baseline/cấu hình cuối luôn chạy lại vì cần trọng số.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess, tempfile
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import Image, display

sys.path.insert(0, os.getcwd())                       # để import data.py, model.py, ... trong code/
QUICK = bool(os.environ.get("LAB_QUICK"))             # chỉ để kiểm tra nhanh code; khi nộp phải là False
RESUME = True                                         # nạp lại results/<exp_id>.json nếu cấu hình không đổi

def find_repo_root():
    d = os.path.abspath(os.getcwd())
    for _ in range(6):                                # đi lên tối đa 6 cấp tìm thư mục có data/ và scripts/
        if os.path.exists(os.path.join(d, "data", "covtype.csv.gz")) and os.path.exists(os.path.join(d, "scripts", "split_data.py")):
            return d
        d = os.path.dirname(d)
    raise FileNotFoundError("Không thấy thư mục gốc repo (chứa data/ và scripts/). Hãy %cd vào submission_<MSSV>/code.")

REPO_ROOT = find_repo_root()                          # = "../.." khi chạy từ submission_<MSSV>/code
OUT_DIR = ".." if os.path.basename(os.getcwd()) == "code" else "./out"
if QUICK: OUT_DIR = "./out_quick"

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "")
print("REPO_ROOT =", REPO_ROOT, "| OUT_DIR =", OUT_DIR, "| QUICK =", QUICK)
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import (DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval,
                   run_evaluate_script, macro_f1_from_confusion)
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx
T_START = time.time()

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train (20%, phân tầng, seed 42), chuẩn hoá 10 cột số bằng thống kê của phần train còn lại, đưa toàn bộ lên device một lần.

In [ ]:
r = subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True, capture_output=True, text=True)
print(r.stdout)
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")

In [ ]:
# Kiểm tra Part 0
Xtr, ytr, Xva, yva, Xev, yev = (data[k].cpu() for k in ("X_tr", "y_tr", "X_val", "y_val", "X_eval", "y_eval"))
assert (len(Xtr), len(Xva), len(Xev)) == (371_847, 92_962, 116_203)
prop = pd.DataFrame({n: np.bincount(y.numpy(), minlength=7) / len(y) for n, y in (("train", ytr), ("val", yva), ("eval", yev))})
print("Tỉ lệ lớp:\n", prop.round(4).T.to_string())
m, s = Xtr[:, :10].mean(0), Xtr[:, :10].std(0)
print(f"\n10 cột số trên X_tr: |mean| lớn nhất = {m.abs().max():.2e}, std trong [{s.min():.4f}, {s.max():.4f}]")
assert m.abs().max() < 1e-3 and (s - 1).abs().max() < 1e-3
print("X_val  mean/std cột số:", f"{Xva[:, :10].mean():.4f} / {Xva[:, :10].std():.4f}", "(≠ 0/1 chính xác vì dùng thống kê của train)")
assert set(torch.unique(Xtr[:, 10:]).tolist()) <= {0.0, 1.0}                      # 44 cột nhị phân giữ nguyên
print("accuracy 'luôn đoán lớp đa số' trên val =", round(data["majority_val_acc"], 4))

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model, ReLU, bias ở mọi lớp, dropout (nếu có) sau ReLU.

In [ ]:
# 1) Dựng M-base, assert số tham số, shape sau từng lớp
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)]
for hid, n in EXPECTED_PARAMS.items():
    assert count_params(MLP(hid)) == n, hid
print(model, "\nsố tham số M-base =", count_params(model), "(M-wide, M-deep cũng khớp bảng)")
h = torch.randn(8, 54, device=device)
with torch.no_grad():
    print("input", tuple(h.shape))
    for layer in model.net:
        h = layer(h); print(f"  {layer.__class__.__name__:7s} -> {tuple(h.shape)}")
assert h.shape == (8, 7)
# He thật sự được áp dụng? std(W1) kỳ vọng sqrt(2/54); mặc định của nn.Linear thì khác
print("std W1 (He)      =", round(model.net[0].weight.std().item(), 4), "| sqrt(2/54) =", round((2 / 54) ** .5, 4))
print("std W1 (default) =", round(MLP((256, 128), init="default").net[0].weight.std().item(), 4), "(≠ He, không dùng làm baseline)")

In [ ]:
# 2) Loss bước 0 trên val (eval mode) ≈ ln 7; cũng đối chiếu macro-F1 tự tính với sklearn
from sklearn.metrics import f1_score
set_seed(0)
m0 = MLP((256, 128), 0.0, "he").to(device)
ev0 = evaluate(m0, data["X_val"], data["y_val"], "ce")
print(f"loss bước 0 (val) = {ev0['loss']:.4f} | ln 7 = {np.log(7):.4f} | chênh = {ev0['loss'] - np.log(7):+.4f}")
p = predict(m0, data["X_val"]).cpu().numpy()
f1_sk = f1_score(data["y_val"].cpu().numpy(), p, average="macro")
print(f"macro-F1 tự tính = {ev0['macro_f1']:.6f} | sklearn = {f1_sk:.6f}")
assert abs(ev0["macro_f1"] - f1_sk) < 1e-9

In [ ]:
# 3) Quá khớp 20 mẫu (tắt dropout): loss phải về gần 0, accuracy 100%
set_seed(0)
idx = torch.randperm(len(data["X_tr"]), generator=torch.Generator().manual_seed(0))[:20].to(device)
xs, ys = data["X_tr"][idx], data["y_tr"][idx]
mo = MLP((256, 128), 0.0, "he").to(device)
opt = build_optimizer("sgd_momentum", mo.parameters(), lr=0.05)
losses = []
for step in range(500):
    mo.train(); loss = F.cross_entropy(mo(xs), ys)
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
acc20 = (mo(xs).argmax(1) == ys).float().mean().item()
print(f"loss: bước 0 = {losses[0]:.4f} -> bước 100 = {losses[100]:.4f} -> bước 500 = {losses[-1]:.5f} | acc 20 mẫu = {acc20:.2f}")
assert losses[-1] < 0.05 and acc20 == 1.0
plt.figure(figsize=(5, 3.2)); plt.semilogy(losses); plt.xlabel("bước"); plt.ylabel("loss (log)")
plt.title("Quá khớp 20 mẫu (SGD+momentum, lr=0.05)"); plt.grid(alpha=.3); plt.show()

In [ ]:
# 4) Gradient có chảy tới mọi tham số không? (một lô val, một lần backward)
mg = MLP((256, 128), 0.0, "he").to(device); mg.train()
loss = F.cross_entropy(mg(data["X_val"][:512]), data["y_val"][:512]); loss.backward()
for (name, p_) in mg.named_parameters():
    assert p_.grad is not None and p_.grad.norm() > 0, name
    print(f"{name:12s} shape {str(tuple(p_.shape)):11s} ‖grad‖ = {p_.grad.norm().item():.4f}")
print("OK: mọi tham số có gradient khác 0")

**Nhận xét Part 1:** *(điền sau khi chạy)* loss bước 0 đo được = ___, so với ln 7 = 1.946 ...; quá khớp 20 mẫu: ...; gradient ...

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình, trả về lịch sử + tóm tắt (+ `best_state`, trọng số của epoch có val loss thấp nhất).
Baseline (`M-base`): SGD + momentum 0.9, cross-entropy, He init, batch 512, **20 epoch**, FP32, không dropout, không clip.
`lr` chọn bằng **val**: quét `lr ∈ {0.003, 0.01, 0.03, 0.1, 0.3}` với seed 1 (đây cũng là thí nghiệm nhóm `hparam`).

In [ ]:
EPOCHS = 2 if QUICK else 20
BASE_CFG = {**DEFAULT_CFG, "epochs": EPOCHS}          # lr đặt sau khi quét
RESULTS, NOTES = {}, {}                               # exp_id -> kết quả (theo thứ tự chạy) / ghi chú cho bảng
SAME_KEYS = ("loss", "optimizer", "lr", "weight_decay", "momentum", "batch", "epochs", "hidden", "dropout",
             "init", "clip_norm", "precision", "scheduler", "seed")
_norm = lambda v: tuple(v) if isinstance(v, (list, tuple)) else v

def run(cfg, need_state=False, note=""):
    """Chạy (hoặc nạp lại) một thí nghiệm, lưu JSON + ảnh figures/<exp_id>.png, in một dòng tóm tắt."""
    cfg = {**BASE_CFG, **cfg}
    eid, path = cfg["exp_id"], f"{OUT_DIR}/results/{cfg['exp_id']}.json"
    res = None
    if RESUME and not need_state and os.path.exists(path):
        old = json.load(open(path))
        if all(_norm(old["cfg"].get(k)) == _norm(cfg.get(k)) for k in SAME_KEYS):
            old["cfg"]["hidden"] = tuple(old["cfg"]["hidden"]); res = old; tag = "(nạp lại)"
    if res is None:
        res = run_experiment(cfg, data); save_result(res, f"{OUT_DIR}/results"); tag = ""
    plot_run(res, f"{OUT_DIR}/figures/{eid}.png")
    RESULTS[eid] = res
    if note: NOTES[eid] = note
    s = res["summary"]
    print(f"{eid:26s} best_ep={s['best_epoch']!s:>4} val_loss={s['best_val_loss']:.4f} acc={s['val_acc']:.4f} "
          f"F1={s['val_macro_f1']:.4f} | {s['time_per_epoch_s']:.1f}s/ep"
          f"{' | DIVERGED' if s['diverged'] else ''} {tag}")
    return res

def table(results, ref=None):
    """Bảng so sánh ngắn; ref = trung bình val_macro_f1 baseline và ngưỡng 2σ (nếu đã có)."""
    rows = []
    for r in results:
        c, s = r["cfg"], r["summary"]
        d = dict(exp_id=c["exp_id"], opt=c["optimizer"], lr=c["lr"], best_ep=s["best_epoch"], val_loss=s["best_val_loss"],
                 val_acc=s["val_acc"], val_f1=s["val_macro_f1"], t_ep=s["time_per_epoch_s"], diverged=s["diverged"])
        if ref: d["ΔF1 vs base"] = s["val_macro_f1"] - ref[0]; d["> 2σ?"] = "Có" if abs(d["ΔF1 vs base"]) > ref[1] else "Không"
        rows.append(d)
    return pd.DataFrame(rows).round(4).to_string(index=False)

# khởi động GPU (không lưu) để thời gian/epoch của lần chạy đầu không bị lệch bởi khởi tạo CUDA
_ = run_experiment({**BASE_CFG, "exp_id": "warmup", "lr": 0.03, "epochs": 1}, data)

In [ ]:
# Quét lr của baseline (SGD+momentum 0.9), chọn bằng val macro-F1
sweep = [run({"exp_id": f"lr-sgdm-{lr:g}", "group": "hparam", "lr": lr, "seed": 1,
              "description": f"Quét lr baseline: SGD+momentum lr={lr:g}"},
             note="Quét lr bằng val để chọn lr baseline") for lr in (0.003, 0.01, 0.03, 0.1, 0.3)]
LR = max(sweep, key=lambda r: np.nan_to_num(r["summary"]["val_macro_f1"], nan=-1))["cfg"]["lr"]
print("\n" + table(sweep)); print(f"\n=> lr baseline chọn bằng VAL (macro-F1 cao nhất) = {LR:g}")
BASE_CFG["lr"] = LR

In [ ]:
# Baseline với 3 seed (cần best_state để dự đoán eval ở Part 4)
base = [run({"exp_id": f"base-s{s}", "group": "baseline", "seed": s, "description": f"Baseline M-base, seed {s}"},
            need_state=True) for s in (1, 2, 3)]
f1s, accs = np.array([r["summary"]["val_macro_f1"] for r in base]), np.array([r["summary"]["val_acc"] for r in base])
F1_MEAN, SIGMA = f1s.mean(), f1s.std(ddof=1)          # ddof=1 giống STDEV của Excel (sheet Seeds)
NOISE = 2 * SIGMA; REF = (F1_MEAN, NOISE)
print(f"\nval macro-F1 = {F1_MEAN:.4f} ± {SIGMA:.4f} | val acc = {accs.mean():.4f} ± {accs.std(ddof=1):.4f} | ngưỡng nhiễu 2σ = {NOISE:.4f}")
print(f"mốc 'đoán đa số': acc {data['majority_val_acc']:.4f}, macro-F1 ≈ 0.094 -> baseline vượt mốc: {accs.min() > data['majority_val_acc']}")
b1 = next(r for r in sweep if r["cfg"]["lr"] == LR)["summary"]["val_macro_f1"]
print(f"kiểm tra tái lập: lr-sgdm-{LR:g} (seed 1) F1 = {b1:.4f} vs base-s1 F1 = {base[0]['summary']['val_macro_f1']:.4f}")
plot_compare(base, ["val_loss", "val_macro_f1", "grad_norm"], f"{OUT_DIR}/figures/compare_baseline.png", "Baseline: 3 seed")
display(Image(f"{OUT_DIR}/figures/base-s1.png")); display(Image(f"{OUT_DIR}/figures/compare_baseline.png"))

**Nhận xét baseline:** *(điền sau khi chạy)* hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số" 0.4876, độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm
Mỗi nhóm: **dự đoán trước (viết trước khi chạy)** → chạy → bảng + ảnh chồng `compare_<nhóm>.png` → **đối chiếu**. Mỗi lần chỉ đổi một yếu tố so với `base-s1` (cùng split, cùng 20 epoch, seed 1), trừ khi ghi rõ.
Cột `ΔF1 vs base` = val macro-F1 − trung bình baseline; "> 2σ?" cho biết chênh lệch có vượt ngưỡng nhiễu seed không.

### Nhóm `hparam` — lr, batch size, độ rộng, độ sâu
**Dự đoán trước (đã viết trước khi chạy):**
- *lr:* với 20 epoch và mạng chưa quá khớp, lr quá nhỏ (0.003) hội tụ chậm nên macro-F1 thấp; lr quá lớn (0.3) bắt đầu dao động; tốt nhất ở giữa (0.03–0.1).
- *batch:* batch 128 cho gấp 4 lần số bước cập nhật trong cùng 20 epoch → F1 cao hơn nhưng mỗi epoch chậm hơn nhiều. Batch 2048 (cùng lr) chỉ còn 1/4 số bước → chưa hội tụ, F1 thấp hơn; tăng lr ×4 (quy tắc tăng lr theo lô) bù lại phần lớn.
- *độ rộng/sâu:* mô hình đang *thiếu khớp* (underfit) nên `M-wide` (3.4× tham số) cải thiện rõ; `M-deep` cải thiện ít hơn.

In [ ]:
hp = [run({"exp_id": "hp-batch128", "group": "hparam", "batch": 128, "description": "batch 128 (4x số bước), lr giữ nguyên"},
          note="Dự đoán: nhiều bước hơn -> F1 cao hơn, chậm hơn mỗi epoch"),
      run({"exp_id": "hp-batch2048", "group": "hparam", "batch": 2048, "description": "batch 2048 (1/4 số bước), lr giữ nguyên"},
          note="Dự đoán: ít bước -> chưa hội tụ, F1 thấp hơn"),
      run({"exp_id": "hp-batch2048-lr4x", "group": "hparam", "batch": 2048, "lr": LR * 4,
           "description": "batch 2048 và lr x4 (quy tắc tăng lr theo lô)"}, note="Đổi 2 yếu tố (batch và lr) có chủ đích; dự đoán: bù lại phần lớn"),
      run({"exp_id": "hp-wide", "group": "hparam", "hidden": (512, 256), "description": "M-wide 54-512-256-7"},
          note="Dự đoán: underfit -> rộng hơn tốt hơn"),
      run({"exp_id": "hp-deep", "group": "hparam", "hidden": (256, 128, 64), "description": "M-deep 54-256-128-64-7"},
          note="Dự đoán: cải thiện ít hơn M-wide")]
print("\n" + table(sweep + hp, REF))
plot_compare(sweep, ["val_loss", "val_macro_f1", "grad_norm"], f"{OUT_DIR}/figures/compare_hparam-lr.png", "hparam: quét lr (SGD+momentum 0.9)")
plot_compare([base[0]] + hp, ["val_loss", "val_macro_f1", "epoch_time_s"], f"{OUT_DIR}/figures/compare_hparam.png", "hparam: batch / độ rộng / độ sâu so với base-s1")
display(Image(f"{OUT_DIR}/figures/compare_hparam-lr.png")); display(Image(f"{OUT_DIR}/figures/compare_hparam.png"))

**Đối chiếu `hparam`:** *(điền sau khi chạy)*

### Nhóm `loss` — cross-entropy vs MSE trên one-hot
**Dự đoán trước:** gradient của CE theo logit là `softmax(z) − y` (giới hạn trong [−1, 1], lớn khi dự đoán sai nặng, không bão hoà); của MSE (`mean` trên mọi phần tử, B×7) là `2(z − y)/7`, nhỏ hơn ~3.5 lần trên mỗi phần tử và không "đẩy" xác suất lớp đúng mạnh khi sai nặng. Nên ở cùng lr, MSE học chậm hơn và macro-F1 thấp hơn (nhất là lớp hiếm); tăng lr (×3, ×10) sẽ bù một phần. Loss CE và MSE khác thang đo nên so bằng accuracy/macro-F1/tốc độ hội tụ, không so loss.

In [ ]:
ls = [run({"exp_id": f"loss-mse-lr{LR * m:g}", "group": "loss", "loss": "mse", "lr": LR * m,
           "description": f"MSE trên one-hot (mean trên B*7 phần tử), lr={LR * m:g}"},
          note="Dự đoán: MSE chậm/kém hơn CE ở cùng lr; lr lớn hơn bù một phần") for m in (1, 3, 10)]
print(table([base[0]] + ls, REF))
plot_compare([base[0]] + ls, ["val_acc", "val_macro_f1", "grad_norm"], f"{OUT_DIR}/figures/compare_loss.png", "loss: CE (base-s1) vs MSE (không so loss trực tiếp)")
display(Image(f"{OUT_DIR}/figures/compare_loss.png"))

**Đối chiếu `loss`:** *(điền sau khi chạy)*

### Nhóm `optimizer` — SGD, SGD+momentum, Adam, AdamW (mỗi bộ thử 3 lr)
**Dự đoán trước:** Adam/AdamW chia bước theo độ lớn gradient từng tham số nên ít nhạy với thang đặc trưng và với lr hơn, và trong 20 epoch hội tụ nhanh hơn SGD+momentum → val macro-F1 ở lr tốt nhất của Adam cao hơn SGD+momentum. AdamW với `weight_decay=0.01` gần như trùng Adam (suy giảm tách riêng quá nhỏ ở 20 epoch). SGD thuần (không momentum) chậm hơn khoảng 10× so với SGD+momentum 0.9 ở cùng lr (bước hiệu dụng `lr/(1−μ)`), nên cần lr lớn hơn. Kết luận phải dựa trên lr tốt nhất của mỗi bộ, không phải một lr duy nhất.

In [ ]:
opt_runs = []
for m in (1, 3, 10):
    opt_runs.append(run({"exp_id": f"opt-sgd-lr{LR * m:g}", "group": "optimizer", "optimizer": "sgd", "lr": LR * m,
                         "description": f"SGD thuần (không momentum) lr={LR * m:g}"}, note="Dự đoán: ~10x chậm hơn SGD+momentum cùng lr"))
for lr in (3e-4, 1e-3, 3e-3):
    opt_runs.append(run({"exp_id": f"opt-adam-lr{lr:g}", "group": "optimizer", "optimizer": "adam", "lr": lr,
                         "description": f"Adam (b1=.9, b2=.999, eps=1e-8) lr={lr:g}"}, note="Dự đoán: nhanh hơn SGD+mom ở lr tốt nhất"))
for lr in (3e-4, 1e-3, 3e-3):
    opt_runs.append(run({"exp_id": f"opt-adamw-lr{lr:g}", "group": "optimizer", "optimizer": "adamw", "lr": lr, "weight_decay": 0.01,
                         "description": f"AdamW weight_decay=0.01 lr={lr:g}"}, note="Dự đoán: gần như trùng Adam"))
pool = {"sgd": [r for r in opt_runs if r["cfg"]["optimizer"] == "sgd"],
        "sgd_momentum": sweep,                         # đã chạy ở Part 2 (nhóm hparam), dùng lại làm tham chiếu
        "adam": [r for r in opt_runs if r["cfg"]["optimizer"] == "adam"],
        "adamw": [r for r in opt_runs if r["cfg"]["optimizer"] == "adamw"]}
best_opt = {k: max(v, key=lambda r: np.nan_to_num(r["summary"]["val_macro_f1"], nan=-1)) for k, v in pool.items()}
print(table(opt_runs, REF))
print("\nLr tốt nhất của từng bộ tối ưu (theo val macro-F1):")
print(table(list(best_opt.values()), REF))
plot_compare(list(best_opt.values()), ["val_loss", "val_macro_f1", "grad_norm"], f"{OUT_DIR}/figures/compare_optimizer.png", "optimizer: lr tốt nhất của mỗi bộ")
plot_compare(opt_runs, ["val_loss", "val_macro_f1", "grad_norm"], f"{OUT_DIR}/figures/compare_optimizer-lr.png", "optimizer: mọi lr đã thử")
display(Image(f"{OUT_DIR}/figures/compare_optimizer.png")); display(Image(f"{OUT_DIR}/figures/compare_optimizer-lr.png"))

**Đối chiếu `optimizer`:** *(điền sau khi chạy)*

### Nhóm `dropout` — q ∈ {0.1, 0.3, 0.5}
**Dự đoán trước:** ở baseline mô hình chưa quá khớp (train loss ≈ val loss, cả hai còn giảm sau 20 epoch), nên dropout không giúp mà làm *thiếu khớp nặng hơn*: q càng lớn, val macro-F1 càng thấp và train loss (đo ở eval mode) càng cao; khoảng cách train–val không đổi đáng kể vì nó vốn đã nhỏ. Dropout chỉ là thuốc khi có quá khớp.

In [ ]:
dr = [run({"exp_id": f"drop-{q:g}", "group": "dropout", "dropout": q, "description": f"Dropout q={q:g} sau mỗi ReLU lớp ẩn"},
          note="Dự đoán: không giúp vì chưa quá khớp; q lớn càng hại") for q in (0.1, 0.3, 0.5)]
print(table([base[0]] + dr, REF))
gap = lambda r: r["summary"]["final_val_loss"] - r["summary"]["final_train_loss"]
print("\nkhoảng cách val_loss − train_loss ở epoch cuối:", {r["cfg"]["exp_id"]: round(gap(r), 4) for r in [base[0]] + dr})
plot_compare([base[0]] + dr, ["train_loss", "val_loss", "val_macro_f1"], f"{OUT_DIR}/figures/compare_dropout.png", "dropout: train/val loss (eval mode) và val macro-F1")
display(Image(f"{OUT_DIR}/figures/compare_dropout.png"))

**Đối chiếu `dropout`:** *(điền sau khi chạy)*

### Nhóm `clipping` — cắt gradient theo chuẩn L2 toàn cục
Ngưỡng `c` chọn từ `grad_norm` của baseline: lấy **trung vị `grad_norm` theo bước** của `base-s1` (`gn_p50`), nên clipping thực sự kích hoạt ở khoảng một nửa số bước.
**Dự đoán trước:** (a) ở lr baseline, clip với `c` = trung vị làm giảm bước hiệu dụng nên không cải thiện, có thể chậm hơn chút (chênh lệch nhỏ, cỡ nhiễu); (b) phản chứng: tăng lr ×10 và ×30 — không clip sẽ dao động/phân kỳ (gai `grad_norm`), còn clip với cùng lr giữ huấn luyện ổn định vì chặn các bước cực lớn; đây mới là tình huống clipping dùng cho.

In [ ]:
C = float(f"{base[0]['summary']['gn_p50']:.2g}")
print(f"baseline: grad_norm theo bước p50 = {base[0]['summary']['gn_p50']:.3f}, p95 = {base[0]['summary']['gn_p95']:.3f}, max = {base[0]['summary']['gn_max']:.3f}  =>  c = {C:g}")
cl = [run({"exp_id": f"clip-c{C:g}-lr{LR:g}", "group": "clipping", "clip_norm": C, "description": f"Clip c={C:g} ở lr baseline"},
          note="Dự đoán: không giúp ở lr bình thường (chênh lệch cỡ nhiễu)")]
for m in (10, 30):
    cl.append(run({"exp_id": f"clip-none-lr{LR * m:g}", "group": "clipping", "lr": LR * m, "description": f"Không clip, lr x{m} = {LR * m:g}"},
                   note=f"Dự đoán: lr x{m} làm huấn luyện dao động/phân kỳ nếu không clip"))
    cl.append(run({"exp_id": f"clip-c{C:g}-lr{LR * m:g}", "group": "clipping", "lr": LR * m, "clip_norm": C,
                   "description": f"Clip c={C:g}, lr x{m} = {LR * m:g}"}, note=f"Dự đoán: clip cứu được huấn luyện ở lr x{m}"))
print("\n" + table([base[0]] + cl, REF))
print("\ntỉ lệ bước bị cắt (clip_frac) / grad_norm max:", {r["cfg"]["exp_id"]: (round(r["summary"]["clip_frac"], 3), round(r["summary"]["gn_max"], 2)) for r in cl})
plot_compare([base[0]] + cl, ["val_loss", "val_macro_f1", "grad_norm"], f"{OUT_DIR}/figures/compare_clipping.png", f"clipping (c={C:g}); grad_norm đo TRƯỚC khi clip")
display(Image(f"{OUT_DIR}/figures/compare_clipping.png"))

**Đối chiếu `clipping`:** *(điền sau khi chạy)*

### Nhóm `amp` — mixed precision (FP16 + GradScaler, BF16)
**Dự đoán trước:** tham số vẫn FP32, autocast chỉ hạ độ chính xác của phép nhân ma trận. Mạng này rất nhỏ (48k tham số, batch 512), thời gian bị chi phối bởi chi phí gọi kernel (launch overhead) chứ không phải FLOPs, và autocast còn thêm các phép ép kiểu → **mixed precision không nhanh hơn, có thể chậm hơn**; bộ nhớ cực đại không giảm đáng kể (dữ liệu nằm sẵn trên GPU); độ chính xác ngang FP32 (trong nhiễu seed). Với `M-wide` mức lợi (nếu có) cũng nhỏ. BF16 trên T4 không có phần cứng native nên có thể chậm hơn FP16.

In [ ]:
amp, AMP_FAIL = [], {}
print("bf16 hỗ trợ:", torch.cuda.is_bf16_supported() if device == "cuda" else "n/a (không có CUDA)")
for tag, hid, ref_name in (("", (256, 128), "base-s1"), ("wide-", (512, 256), "hp-wide")):
    for prec in ("fp16", "bf16"):
        eid = f"amp-{tag}{prec}"
        try:
            amp.append(run({"exp_id": eid, "group": "amp", "precision": prec, "hidden": hid,
                            "description": f"{prec} autocast{' + GradScaler' if prec == 'fp16' else ''}, {'-'.join(map(str, hid))}; so với {ref_name} (FP32)"},
                           note="Dự đoán: không nhanh hơn (overhead kernel), độ chính xác ngang FP32"))
        except Exception as e:
            AMP_FAIL[eid] = repr(e)[:200]; print(f"{eid}: KHÔNG CHẠY ĐƯỢC -> {AMP_FAIL[eid]}")
cmp_ = [base[0], RESULTS["hp-wide"]] + amp
print("\n" + pd.DataFrame([dict(exp_id=r["cfg"]["exp_id"], precision=r["cfg"]["precision"], hidden="-".join(map(str, r["cfg"]["hidden"])),
                                val_f1=r["summary"]["val_macro_f1"], t_per_epoch_s=r["summary"]["time_per_epoch_s"],
                                peak_mem_MB=r["summary"]["peak_mem_MB"]) for r in cmp_]).round(4).to_string(index=False))
plot_compare(cmp_, ["val_macro_f1", "epoch_time_s", "grad_norm"], f"{OUT_DIR}/figures/compare_amp.png", "amp: FP32 vs FP16 vs BF16 (M-base và M-wide)")
display(Image(f"{OUT_DIR}/figures/compare_amp.png"))

**Đối chiếu `amp`:** *(điền sau khi chạy)*

### Nhóm `init` — khởi tạo tham số (baseline dùng He)
**Dự đoán trước:** (a) `zeros`: mọi nơ-ron trong một lớp giống hệt nhau và ReLU(0)=0 nên gradient tới W1, W2, W3 đều bằng 0 (chỉ bias lớp ra có gradient) → mạng không học, chỉ học tần suất lớp: val acc ≈ 0.4876 và macro-F1 ≈ 0.094 (mốc "đoán đa số"), loss bước 0 = ln 7. (b) `normal` (std 0.01): kích hoạt co dần về 0 qua các lớp (std nhỏ) nên gradient của các lớp dưới rất nhỏ → học chậm lúc đầu, kém He sau 20 epoch. (c) `xavier` và `default` (mạng chỉ 3 lớp, nông): gần He, chênh lệch cỡ nhiễu. Loss bước 0 ≈ ln 7 với mọi cách vì logits nhỏ.

In [ ]:
ini = [run({"exp_id": f"init-{n}", "group": "init", "init": n, "description": f"Khởi tạo {n} (bias=0, trừ default)"},
           note={"zeros": "Dự đoán: không học (đối xứng + ReLU(0)), F1≈0.094", "normal": "Dự đoán: học chậm (kích hoạt co về 0)",
                 "xavier": "Dự đoán: gần He", "default": "Dự đoán: gần He"}[n]) for n in ("zeros", "normal", "xavier", "default")]
tab = [dict(init=r["cfg"]["init"], step0_loss=r["summary"]["step0_loss"],
            act_std_L1=r["summary"]["act_std_step0"][0], act_std_L2=r["summary"]["act_std_step0"][1], act_std_logits=r["summary"]["act_std_step0"][2],
            final_grad_norm=r["history"]["grad_norm"][-1], val_acc=r["summary"]["val_acc"], val_f1=r["summary"]["val_macro_f1"]) for r in [base[0]] + ini]
tab[0]["init"] = "he (base-s1)"
print(pd.DataFrame(tab).round(4).to_string(index=False))
print("\n(act_std_* = độ lệch chuẩn đầu ra của mỗi nn.Linear ở bước 0 trên 512 mẫu val; mẫu đầu vào có std ≈", round(data["X_val"][:512].std().item(), 3), ")")
plot_compare([base[0]] + ini, ["val_loss", "val_macro_f1", "grad_norm"], f"{OUT_DIR}/figures/compare_init.png", "init: He (base-s1) vs zeros / normal / xavier / default")
display(Image(f"{OUT_DIR}/figures/compare_init.png"))

**Đối chiếu `init`:** *(điền sau khi chạy)*

## Part 4 — Cấu hình cuối cùng, đánh giá trên eval, bảng và nộp bài
**Chọn cấu hình cuối CHỈ bằng val.** Bộ tối ưu + lr lấy từ cấu hình `M-base` (batch 512, CE, He, FP32) có val macro-F1 cao nhất trong các lần chạy đã có; sau đó thử 3 ứng viên kết hợp (M-wide, +cosine, +epoch gấp đôi) trên val, chọn ứng viên có val macro-F1 cao nhất rồi chạy 3 seed. Cấu hình cuối đổi nhiều yếu tố cùng lúc (ghi rõ trong `description`).

In [ ]:
okf = lambda r: np.nan_to_num(r["summary"]["val_macro_f1"], nan=-1)
cand_pool = [r for r in RESULTS.values() if r["cfg"]["hidden"] == (256, 128) and r["cfg"]["batch"] == 512 and r["cfg"]["dropout"] == 0
             and r["cfg"]["clip_norm"] is None and r["cfg"]["precision"] == "fp32" and r["cfg"]["init"] == "he"
             and r["cfg"]["loss"] == "ce" and r["cfg"]["group"] in ("baseline", "optimizer", "hparam")]
top = max(cand_pool, key=okf)["cfg"]
OPT = {k: top[k] for k in ("optimizer", "lr", "weight_decay")}
print(f"bộ tối ưu chọn bằng val: {OPT} (từ {top['exp_id']})")
LONG = 2 * EPOCHS
fcands = [run({"exp_id": "final-cand-wide", "group": "final", **OPT, "hidden": (512, 256),
               "description": f"Ứng viên: {top['optimizer']} + M-wide, {EPOCHS} epoch"}, note="Ứng viên cho cấu hình cuối (chọn bằng val)"),
          run({"exp_id": "final-cand-wide-cos", "group": "final", **OPT, "hidden": (512, 256), "scheduler": "cosine",
               "description": f"Ứng viên: {top['optimizer']} + M-wide + cosine, {EPOCHS} epoch"}, note="Ứng viên cho cấu hình cuối (chọn bằng val)"),
          run({"exp_id": f"final-cand-wide-cos-{LONG}ep", "group": "final", **OPT, "hidden": (512, 256), "scheduler": "cosine", "epochs": LONG,
               "description": f"Ứng viên: {top['optimizer']} + M-wide + cosine, {LONG} epoch"}, note="Ứng viên cho cấu hình cuối (chọn bằng val)")]
print("\n" + table(fcands + [base[0]], REF))
W = max(fcands, key=okf)["cfg"]
FINAL = {k: W[k] for k in ("optimizer", "lr", "weight_decay", "hidden", "scheduler", "epochs", "batch", "dropout", "init", "loss", "clip_norm", "precision")}
print("\ncấu hình cuối cùng (chọn bằng val):", FINAL)
final = [run({"exp_id": f"final-s{s}", "group": "final", **FINAL, "seed": s,
              "description": f"CẤU HÌNH CUỐI, seed {s}: {FINAL['optimizer']} lr={FINAL['lr']:g}, {'-'.join(map(str, FINAL['hidden']))}, {FINAL['scheduler'] or 'lr cố định'}, {FINAL['epochs']} epoch (đổi nhiều yếu tố so với baseline)"},
             need_state=True, note="Cấu hình cuối; đổi nhiều yếu tố so với baseline") for s in (1, 2, 3)]
fF = np.array([r["summary"]["val_macro_f1"] for r in final])
print(f"\nval macro-F1: final = {fF.mean():.4f} ± {fF.std(ddof=1):.4f} | baseline = {F1_MEAN:.4f} ± {SIGMA:.4f} | Δ = {fF.mean() - F1_MEAN:+.4f} (2σ_base = {NOISE:.4f})")
plot_compare(base + final, ["val_loss", "val_macro_f1", "grad_norm"], f"{OUT_DIR}/figures/compare_final.png", "final (3 seed) vs baseline (3 seed)")
display(Image(f"{OUT_DIR}/figures/compare_final.png"))

### Đánh giá trên tập eval (chỉ làm sau khi đã chọn xong cấu hình bằng val)
Nạp `best_state` (epoch val loss thấp nhất), dự đoán TOÀN BỘ `X_eval` ở `eval()`, ghi CSV rồi chấm bằng đúng `scripts/evaluate.py`.
`predictions_eval.csv` và `eval_result.json` nộp là của **`final-s1`**. Các seed còn lại (baseline và final) chỉ để đo nhiễu của điểm eval; dự đoán của chúng ghi tạm rồi xoá, kết quả lưu ở `results/eval_<exp_id>.json`.
Không chỉnh lại cấu hình sau khi thấy điểm eval.

In [ ]:
EVAL = {}
pred_path, out_json = f"{OUT_DIR}/predictions_eval.csv", f"{OUT_DIR}/eval_result.json"
final_eval(RESULTS["final-s1"]["cfg"], RESULTS["final-s1"], data, pred_path)
print("=== final-s1 (nộp) ===")
EVAL["final-s1"] = run_evaluate_script(pred_path, out_json, REPO_ROOT)
for eid in ("base-s1", "base-s2", "base-s3", "final-s2", "final-s3"):
    with tempfile.TemporaryDirectory() as td:
        final_eval(RESULTS[eid]["cfg"], RESULTS[eid], data, f"{td}/p.csv")
        EVAL[eid] = run_evaluate_script(f"{td}/p.csv", f"{OUT_DIR}/results/eval_{eid}.json", REPO_ROOT, verbose=False)
ev = pd.DataFrame({k: dict(eval_acc=v["accuracy"], eval_macro_f1=v["macro_f1"], val_macro_f1=RESULTS[k]["summary"]["val_macro_f1"],
                           val_acc=RESULTS[k]["summary"]["val_acc"]) for k, v in EVAL.items()}).T.loc[["base-s1", "base-s2", "base-s3", "final-s1", "final-s2", "final-s3"]]
print("\n" + ev.round(4).to_string())
for name in ("base", "final"):
    x = ev.loc[[i for i in ev.index if i.startswith(name)], "eval_macro_f1"]
    print(f"{name}: eval macro-F1 = {x.mean():.4f} ± {x.std(ddof=1):.4f}")
d = ev.loc[[i for i in ev.index if i.startswith('final')], "eval_macro_f1"].mean() - ev.loc[[i for i in ev.index if i.startswith('base')], "eval_macro_f1"].mean()
print(f"cải thiện eval macro-F1 (final − base, TB 3 seed) = {d:+.4f}; riêng final-s1 − base-s1 = {ev.loc['final-s1','eval_macro_f1'] - ev.loc['base-s1','eval_macro_f1']:+.4f}")
print(f"chênh val − eval (macro-F1), final-s1: {ev.loc['final-s1','val_macro_f1'] - ev.loc['final-s1','eval_macro_f1']:+.4f}")

In [ ]:
# Phân tích lỗi theo lớp (final-s1 so với base-s1)
NAMES = ["Spruce/Fir", "Lodgepole", "Ponderosa", "Cottonwood", "Aspen", "Douglas-fir", "Krummholz"]
def per_class(e):
    return pd.DataFrame(e["per_class"]).set_index("cls")[["support", "precision", "recall", "f1"]].set_axis(NAMES)
pcf, pcb = per_class(EVAL["final-s1"]), per_class(EVAL["base-s1"])
print("final-s1 (nộp) — chỉ số theo lớp trên eval:\n", pcf.round(4).to_string())
print("\nbase-s1 — F1 theo lớp:", pcb["f1"].round(4).to_dict())
cm = np.array(EVAL["final-s1"]["confusion_matrix"]); cmb = np.array(EVAL["base-s1"]["confusion_matrix"])
worst = int(pcf["f1"].values.argmin()); row = cm[worst] / cm[worst].sum()
print(f"\nlớp khó nhất: {worst} ({NAMES[worst]}), F1 = {pcf['f1'].iloc[worst]:.4f}; tỉ lệ dự đoán của các mẫu thuộc lớp này:",
      {NAMES[j]: round(row[j], 3) for j in np.argsort(-row)[:3]})
off = [(cm[i, j], NAMES[i], NAMES[j]) for i in range(7) for j in range(7) if i != j]
print("5 cặp nhầm lẫn nhiều nhất (thật -> dự đoán):", [(f"{a}->{b}", int(n)) for n, a, b in sorted(off, reverse=True)[:5]])
fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
for a, (c_, t) in zip(ax, ((cmb, "base-s1"), (cm, "final-s1 (nộp)"))):
    im = a.imshow(c_ / c_.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
    a.set_xticks(range(7)); a.set_xticklabels(NAMES, rotation=45, ha="right"); a.set_yticks(range(7)); a.set_yticklabels(NAMES)
    a.set_xlabel("dự đoán"); a.set_ylabel("nhãn thật"); a.set_title(f"Ma trận nhầm lẫn (chuẩn hoá theo hàng), eval: {t}")
    for i in range(7):
        for j in range(7): a.text(j, i, f"{c_[i, j] / c_[i].sum():.2f}", ha="center", va="center", fontsize=7, color="white" if c_[i, j] / c_[i].sum() > .5 else "black")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/figures/compare_confusion.png", dpi=110, bbox_inches="tight"); plt.show()

**Phân tích lỗi:** *(điền sau khi chạy)* lớp khó nhất, nhầm với lớp nào, vì sao (số mẫu, đặc trưng tương đồng, mất cân bằng) ...

In [ ]:
# Bảng experiments.xlsx từ results/*.json (đọc lại từ đĩa để chắc chắn JSON đủ thông tin)
SUMMARY_NOTES = {}      # nhận xét ngắn theo nhóm cho sheet Summary (điền sau khi có kết quả)
disk = {r["cfg"]["exp_id"]: r for r in load_results(f"{OUT_DIR}/results")}
rows = [to_row(disk[eid], EVAL.get(eid), NOTES.get(eid, "")) for eid in RESULTS if eid in disk]
write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx",
           seed_ids=["base-s1", "base-s2", "base-s3"], summary_notes=SUMMARY_NOTES)
figs = sorted(f for f in os.listdir(f"{OUT_DIR}/figures") if f.endswith(".png") and not f.startswith("compare_"))
assert sorted(r["figure_file"].split("/")[-1] for r in rows) == figs, "mỗi dòng bảng phải có đúng một ảnh figures/<exp_id>.png"
print(f"experiments.xlsx: {len(rows)} dòng, {len(figs)} ảnh thí nghiệm, {len([f for f in os.listdir(f'{OUT_DIR}/figures') if f.startswith('compare_')])} ảnh compare_*")
print(pd.Series([r["group"] for r in rows]).value_counts().to_dict())
print(f"Tổng thời gian notebook: {(time.time() - T_START) / 60:.1f} phút")

**Hết notebook.** Xem `../REPORT.md` cho báo cáo kết luận.